# 08 Multi-Pin Pilot Analysis

Analyze rows where one coordinate is insufficient.

In [8]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.metrics import haversine_meters

PROCESSED = PROJECT_ROOT / "data" / "processed"
path = PROCESSED / "multipin_pilot.csv"

df = pd.read_csv(path)
df.shape


(21, 39)

In [9]:
df.head()


,pilot_source,id,name,category_primary,region,tier_label,place_complexity,pin_ambiguity,manual_review_status,manual_should_move,...,accessible_entry_lat,accessible_entry_lon,accessible_entry_confidence,multipin_notes,manual_notes,should_move,tier,gt_model,offset_euclidean_m,offset_manhattan_m
0,high_offset,08f44f082e49c142032785383719bd54,Starke / Gainesville N.E. KOA Holiday,campground,FL,open_space,complex,high,ambiguous,true,...,NaN,NaN,NaN,PROXY LABEL: using LLM ground-truth coordinate...,"Open-space place; main vehicle access, gate, p...",True,3,gpt-4o-mini,41.401470,41.401470
1,high_offset,08f441ad446d54200323a1a9871a5d0d,Pinch A Penny Pool Patio Spa,hot_tubs_and_pools,FL,multi_tenant,simple,low,ambiguous,true,...,NaN,NaN,NaN,PROXY LABEL: using LLM ground-truth coordinate...,Multi-tenant or unit-level place; storefront c...,True,2,gpt-4o-mini,43.570571,52.837082
2,high_offset,08f288423356c91103ca007f24b8f280,Cafe Rio Fresh Modern Mexican,fast_food_restaurant,ID,standard_commercial,multi_tenant,medium,ambiguous,true,...,NaN,NaN,NaN,PROXY LABEL: using LLM ground-truth coordinate...,Multi-tenant or unit-level place; storefront c...,True,1,gpt-4o-mini,35.647161,43.228537
3,high_offset,08f2a9dac8286a54032f3c8132bc29e9,New River Hideaways,holiday_rental_home,WV,open_space,simple,low,ambiguous,true,...,NaN,NaN,NaN,PROXY LABEL: using LLM ground-truth coordinate...,"Open-space place; main vehicle access, gate, p...",True,3,gpt-4o-mini,39.083332,47.395506
4,high_offset,08f279a0d2a9ace503c8225af210ef6a,Dead Indian Campground,campground,NaN,open_space,complex,high,ambiguous,true,...,NaN,NaN,NaN,PROXY LABEL: using LLM ground-truth coordinate...,"Open-space place; main vehicle access, gate, p...",True,3,gpt-4o-mini,33.925882,33.925882


In [10]:
pin_cols = [
    "pedestrian_entry_lat",
    "pedestrian_entry_lon",
    "vehicle_entry_lat",
    "vehicle_entry_lon",
    "delivery_entry_lat",
    "delivery_entry_lon",
    "accessible_entry_lat",
    "accessible_entry_lon",
]

df[pin_cols].notna().sum()


pedestrian_entry_lat    13
pedestrian_entry_lon    13
vehicle_entry_lat       14
vehicle_entry_lon       14
delivery_entry_lat       0
delivery_entry_lon       0
accessible_entry_lat     0
accessible_entry_lon     0
dtype: int64

In [11]:
def has_pair(row, lat_col, lon_col):
    return pd.notna(row.get(lat_col)) and pd.notna(row.get(lon_col))

df["has_pedestrian_entry"] = df.apply(lambda r: has_pair(r, "pedestrian_entry_lat", "pedestrian_entry_lon"), axis=1)
df["has_vehicle_entry"] = df.apply(lambda r: has_pair(r, "vehicle_entry_lat", "vehicle_entry_lon"), axis=1)
df["has_delivery_entry"] = df.apply(lambda r: has_pair(r, "delivery_entry_lat", "delivery_entry_lon"), axis=1)
df["has_accessible_entry"] = df.apply(lambda r: has_pair(r, "accessible_entry_lat", "accessible_entry_lon"), axis=1)

df[["has_pedestrian_entry", "has_vehicle_entry", "has_delivery_entry", "has_accessible_entry"]].sum()


has_pedestrian_entry    13
has_vehicle_entry       14
has_delivery_entry       0
has_accessible_entry     0
dtype: int64

In [12]:
def distance_between(row, a_lat, a_lon, b_lat, b_lon):
    if not all(pd.notna(row.get(c)) for c in [a_lat, a_lon, b_lat, b_lon]):
        return np.nan
    return haversine_meters(row[a_lat], row[a_lon], row[b_lat], row[b_lon])

df["pedestrian_vehicle_distance_m"] = df.apply(
    lambda r: distance_between(r, "pedestrian_entry_lat", "pedestrian_entry_lon", "vehicle_entry_lat", "vehicle_entry_lon"),
    axis=1,
)

df["current_to_pedestrian_m"] = df.apply(
    lambda r: distance_between(r, "current_lat", "current_lon", "pedestrian_entry_lat", "pedestrian_entry_lon"),
    axis=1,
)

df["current_to_vehicle_m"] = df.apply(
    lambda r: distance_between(r, "current_lat", "current_lon", "vehicle_entry_lat", "vehicle_entry_lon"),
    axis=1,
)

df[["pedestrian_vehicle_distance_m", "current_to_pedestrian_m", "current_to_vehicle_m"]].describe()


,pedestrian_vehicle_distance_m,current_to_pedestrian_m,current_to_vehicle_m
count,6.000000,13.000000,14.000000
mean,44.279178,20.436544,19.055795
std,23.400940,27.496081,19.879337
min,18.721384,0.000000,0.000000
25%,36.210102,0.000000,0.000000
50%,39.633220,0.000000,16.943882
75%,42.953054,38.019077,38.302682
max,88.558515,88.558515,41.404199


In [13]:
pd.crosstab(df["tier_label"], df["manual_primary_pin_type"], dropna=False)


manual_primary_pin_type,pedestrian_entry,vehicle_entry
tier_label,,
multi_tenant,10,0
open_space,0,8
standard_commercial,3,0


In [14]:
summary_lines = [
    "Multi-Pin Pilot Analysis Summary",
    "",
    f"Rows: {len(df)}",
    "",
    "Pin coverage",
    df[["has_pedestrian_entry", "has_vehicle_entry", "has_delivery_entry", "has_accessible_entry"]].sum().to_string(),
    "",
    "Pedestrian/vehicle distance",
    df["pedestrian_vehicle_distance_m"].describe().to_string(),
    "",
    "Current to pedestrian",
    df["current_to_pedestrian_m"].describe().to_string(),
    "",
    "Current to vehicle",
    df["current_to_vehicle_m"].describe().to_string(),
]

summary_path = PROCESSED / "multipin_pilot_analysis_summary.txt"
summary_path.write_text("\n".join(summary_lines) + "\n")
df.to_csv(PROCESSED / "multipin_pilot_analyzed.csv", index=False)

summary_path


WindowsPath('c:/Users/aaron/Documents/Pin-To-Place/data/processed/multipin_pilot_analysis_summary.txt')